%md
| Transformation Area     | Columns                              | Transformation                  |
| ----------------------- | ------------------------------------ | ------------------------------- |
| Type Casting            | quantity, price, tax, discount       | Cast to numeric types           |
| Date Standardization    | sale_timestamp, sale_date            | TIMESTAMP / DATE                |
| Text Cleaning           | country, city, state, product, store | TRIM + INITCAP                  |
| Payment Standardization | payment_method                       | Map to canonical values         |
| Status Standardization  | order_status                         | Map to canonical statuses       |
| Null Validation         | IDs, required fields                 | Flag missing values             |
| Quantity Validation     | quantity                             | Must be > 0                     |
| Price Validation        | unit_price                           | Must be > 0                     |
| Discount Validation     | discount                             | 0–100                           |
| Tax Validation          | tax                                  | 0–100                           |
| Currency Validation     | currency                             | Allowed currency list           |
| Shipping Validation     | shipping_cost                        | >= 0                            |
| Cost Validation         | cost_price                           | >= 0                            |
| Return Validation       | return_quantity                      | 0 ≤ return qty ≤ quantity       |
| Return Flag Validation  | return_flag                          | Consistent with return quantity |
| Weather Validation      | temperature_c                        | Valid temperature range         |
| Exchange Validation     | exchange_rate                        | > 0                             |
| Amount Calculation      | quantity, unit_price, discount, tax  | Calculate expected amount       |
| Amount Reconciliation   | total_amount                         | Compare source vs calculated    |
| Profit Reconciliation   | profit_amount                        | Validate calculated profit      |
| Deduplication           | sale_id                              | Keep latest/valid record        |
| Data Quality            | All validation flags                 | Create `is_valid`               |
| Error Handling          | Invalid records                      | Quarantine to rejected table    |
| Technical Lineage       | run_id, source_file                  | Preserve source lineage         |
| Processing Metadata     | ingestion timestamp                  | Add Silver processing timestamp |




"I classified Silver transformations into structural transformations, standardization, data-quality validation, business-rule validation, reconciliation, and deduplication. Invalid records are quarantined with rejection reasons so that no source data is silently lost."

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# ============================================================
# SILVER NOTEBOOK PARAMETERS
# ============================================================

dbutils.widgets.text("catalog", "sales")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

print(f"Catalog       : {catalog}")
print(f"Bronze Schema : {bronze_schema}")
print(f"Silver Schema : {silver_schema}")

In [0]:
# ============================================================
# CONFIGURATION
# ============================================================

BRONZE_TABLE = f"{catalog}.{bronze_schema}.sales"

SILVER_TABLE = f"{catalog}.{silver_schema}.sales"

REJECTED_TABLE = f"{catalog}.{silver_schema}.sales_rejected"

print(f"Bronze  : {BRONZE_TABLE}")
print(f"Silver  : {SILVER_TABLE}")
print(f"Rejected: {REJECTED_TABLE}")

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}
""")

In [0]:
# ============================================================
# READ BRONZE
# ============================================================

bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze records: {bronze_df.count()}")

In [0]:
# ============================================================
# STANDARDIZE DATA TYPES
# ============================================================

silver_df = (
    bronze_df

    .withColumn(
        "sale_timestamp",
        F.expr("try_cast(sale_timestamp AS TIMESTAMP)")
    )

    .withColumn(
        "sale_date",
        F.expr("try_cast(sale_date AS DATE)")
    )

    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )

    .withColumn(
        "unit_price",
        F.col("unit_price").cast("double")
    )

    .withColumn(
        "discount",
        F.col("discount").cast("double")
    )

    .withColumn(
        "tax",
        F.col("tax").cast("double")
    )

    .withColumn(
        "total_amount",
        F.col("total_amount").cast("double")
    )

    .withColumn(
        "shipping_cost",
        F.col("shipping_cost").cast("double")
    )

    .withColumn(
        "cost_price",
        F.col("cost_price").cast("double")
    )

    .withColumn(
        "profit_amount",
        F.col("profit_amount").cast("double")
    )

    .withColumn(
        "temperature_c",
        F.col("temperature_c").cast("double")
    )

    .withColumn(
        "exchange_rate",
        F.col("exchange_rate").cast("double")
    )
)

In [0]:
# ============================================================
# TEXT STANDARDIZATION
# ============================================================

text_columns = [
    "country",
    "region",
    "state",
    "city",
    "store_name",
    "product_name",
    "category",
    "subcategory",
    "payment_method",
    "customer_type",
    "currency",
    "sales_channel",
    "order_status",
    "weather_condition"
]

for column_name in text_columns:
    silver_df = silver_df.withColumn(
        column_name,
        F.trim(F.col(column_name))
    )

In [0]:
silver_df = (
    silver_df
    .withColumn("country", F.initcap("country"))
    .withColumn("city", F.initcap("city"))
    .withColumn("state", F.initcap("state"))
    .withColumn("category", F.initcap("category"))
    .withColumn("customer_type", F.initcap("customer_type"))
    .withColumn("payment_method", F.initcap("payment_method"))
)

In [0]:
# ============================================================
# PAYMENT STANDARDIZATION
# ============================================================

silver_df = silver_df.withColumn(
    "payment_method",
    F.when(
        F.upper(F.col("payment_method")) == "UPI",
        "UPI"
    )
    .when(
        F.upper(F.col("payment_method")) == "CREDIT CARD",
        "Credit Card"
    )
    .when(
        F.upper(F.col("payment_method")) == "DEBIT CARD",
        "Debit Card"
    )
    .when(
        F.upper(F.col("payment_method")) == "CASH",
        "Cash"
    )
    .when(
        F.upper(F.col("payment_method")) == "WALLET",
        "Wallet"
    )
    .when(
        F.upper(F.col("payment_method")) == "NET BANKING",
        "Net Banking"
    )
    .otherwise(None)
)

In [0]:
# ============================================================
# ORDER STATUS STANDARDIZATION
# ============================================================

silver_df = silver_df.withColumn(
    "order_status",
    F.when(
        F.upper(F.col("order_status")) == "COMPLETED",
        "Completed"
    )
    .when(
        F.upper(F.col("order_status")) == "CANCELLED",
        "Cancelled"
    )
    .when(
        F.upper(F.col("order_status")) == "RETURNED",
        "Returned"
    )
    .when(
        F.upper(F.col("order_status")) == "PENDING",
        "Pending"
    )
    .otherwise(None)
)

In [0]:
# ============================================================
# EXPECTED AMOUNT
# ============================================================

silver_df = silver_df.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
)

silver_df = silver_df.withColumn(
    "discount_amount",
    F.col("gross_amount") *
    F.col("discount") / 100
)

silver_df = silver_df.withColumn(
    "net_amount_before_tax",
    F.col("gross_amount") -
    F.col("discount_amount")
)

silver_df = silver_df.withColumn(
    "tax_amount",
    F.col("net_amount_before_tax") *
    F.col("tax") / 100
)

silver_df = silver_df.withColumn(
    "expected_total_amount",
    F.round(
        F.col("net_amount_before_tax") +
        F.col("tax_amount"),
        2
    )
)

In [0]:
# ============================================================
# AMOUNT VALIDATION
# ============================================================

silver_df = silver_df.withColumn(
    "amount_valid",
    F.when(
        F.col("total_amount").isNull(),
        False
    )
    .when(
        F.abs(
            F.col("total_amount") -
            F.col("expected_total_amount")
        ) <= 1,
        True
    )
    .otherwise(False)
)

In [0]:
# ============================================================
# DATA QUALITY FLAGS
# ============================================================

silver_df = (
    silver_df

    .withColumn(
        "valid_sale_id",
        F.col("sale_id").isNotNull()
    )

    .withColumn(
        "valid_quantity",
        (F.col("quantity") > 0)
    )

    .withColumn(
        "valid_unit_price",
        (F.col("unit_price") > 0)
    )

    .withColumn(
        "valid_discount",
        (
            (F.col("discount") >= 0) &
            (F.col("discount") <= 100)
        )
    )

    .withColumn(
        "valid_tax",
        (
            (F.col("tax") >= 0) &
            (F.col("tax") <= 100)
        )
    )

    .withColumn(
        "valid_customer",
        F.col("customer_id").isNotNull()
    )

    .withColumn(
        "valid_product",
        F.col("product_id").isNotNull()
    )

    .withColumn(
    "valid_product_name",
    F.col("product_name").isNotNull()
    )

    .withColumn(
        "valid_store",
        F.col("store_id").isNotNull()
    )

    .withColumn(
        "valid_country",
        F.col("country").isNotNull()
    )

    .withColumn(
        "valid_city",
        F.col("city").isNotNull()
    )

    .withColumn(
        "valid_currency",
        F.col("currency").isin(
            "INR",
            "USD",
            "EUR",
            "GBP",
            "AED",
            "SGD"
        )
    )

    .withColumn(
        "valid_payment",
        F.col("payment_method").isNotNull()
    )

    .withColumn(
        "valid_status",
        F.col("order_status").isNotNull()
    )

    .withColumn(
    "valid_store_name",
    F.col("store_name").isNotNull()
)

    .withColumn(
        "valid_shipping",
        F.col("shipping_cost") >= 0
    )

    .withColumn(
        "valid_cost_price",
        F.col("cost_price") >= 0
    )

    .withColumn(
        "valid_temperature",
        (
            (F.col("temperature_c") >= -50) &
            (F.col("temperature_c") <= 60)
        )
    )

    .withColumn(
        "valid_exchange_rate",
        F.col("exchange_rate") > 0
    )

    .withColumn(
        "valid_return_quantity",
        F.col("return_quantity") >= 0
    )

    .withColumn(
        "valid_amount",
        F.col("amount_valid")
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "valid_return_quantity",
    (
        (F.col("return_quantity") >= 0) &
        (F.col("return_quantity") <= F.col("quantity"))
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "valid_return_flag",
    F.when(
        F.col("return_flag") == True,
        F.col("return_quantity") > 0
    )
    .otherwise(
        F.col("return_quantity") == 0
    )
)

In [0]:
# ============================================================
# OVERALL DATA QUALITY
# ============================================================

validation_columns = [
    "valid_sale_id",
    "valid_quantity",
    "valid_unit_price",
    "valid_discount",
    "valid_tax",
    "valid_customer",
    "valid_product",
    "valid_store",
    "valid_country",
    "valid_city",
    "valid_currency",
    "valid_payment",
    "valid_status",
    "valid_shipping",
    "valid_cost_price",
    "valid_temperature",
    "valid_exchange_rate",
    "valid_return_quantity",
    "valid_return_flag",
    "valid_amount",
    "valid_store_name",
    "valid_product_name"
]

silver_df = silver_df.withColumn(
    "is_valid",
    F.expr(
        " AND ".join(
            [f"COALESCE({c}, false)" for c in validation_columns]
        )
    )
)

In [0]:
%skip
display(silver_df)

In [0]:
# ============================================================
# REJECTION REASON
# example: If valid_quantity is false or NULL, return "INVALID_QUANTITY".
# ============================================================

silver_df = silver_df.withColumn(
    "rejection_reason",
    F.concat_ws(
        "; ",

        F.when(
            ~F.coalesce(F.col("valid_sale_id"), F.lit(False)),
            "INVALID_SALE_ID"
        ),

        F.when(
            ~F.coalesce(F.col("valid_product_name"), F.lit(False)),
            "INVALID_PRODUCT_NAME"
        ),

        F.when(
            ~F.coalesce(F.col("valid_quantity"), F.lit(False)),
            "INVALID_QUANTITY"
        ),

        F.when(
            ~F.coalesce(F.col("valid_unit_price"), F.lit(False)),
            "INVALID_UNIT_PRICE"
        ),

        F.when(
            ~F.coalesce(F.col("valid_discount"), F.lit(False)),
            "INVALID_DISCOUNT"
        ),

        F.when(
            ~F.coalesce(F.col("valid_tax"), F.lit(False)),
            "INVALID_TAX"
        ),

        F.when(
            ~F.coalesce(F.col("valid_customer"), F.lit(False)),
            "MISSING_CUSTOMER"
        ),

        F.when(
            ~F.coalesce(F.col("valid_product"), F.lit(False)),
            "MISSING_PRODUCT"
        ),

        F.when(
            ~F.coalesce(F.col("valid_store"), F.lit(False)),
            "MISSING_STORE"
        ),

        F.when(
            ~F.coalesce(F.col("valid_currency"), F.lit(False)),
            "INVALID_CURRENCY"
        ),

        F.when(
            ~F.coalesce(F.col("valid_payment"), F.lit(False)),
            "INVALID_PAYMENT_METHOD"
        ),

        F.when(
            ~F.coalesce(F.col("valid_status"), F.lit(False)),
            "INVALID_ORDER_STATUS"
        ),

        F.when(
            ~F.coalesce(F.col("valid_shipping"), F.lit(False)),
            "INVALID_SHIPPING_COST"
        ),

        F.when(
            ~F.coalesce(F.col("valid_temperature"), F.lit(False)),
            "INVALID_TEMPERATURE"
        ),

        F.when(
            ~F.coalesce(F.col("valid_exchange_rate"), F.lit(False)),
            "INVALID_EXCHANGE_RATE"
        ),

        F.when(
            ~F.coalesce(F.col("valid_return_quantity"), F.lit(False)),
            "INVALID_RETURN_QUANTITY"
        ),

        F.when(
            ~F.coalesce(F.col("valid_store_name"), F.lit(False)),
            "INVALID_STORE_NAME"
        ),

        F.when(
            ~F.coalesce(F.col("valid_amount"), F.lit(False)),
            "INVALID_TOTAL_AMOUNT"
        )
    )
)

In [0]:
%skip
display(silver_df)

In [0]:
# ============================================================
# DUPLICATE DETECTION
# ============================================================

window_spec = (
    Window
    .partitionBy("sale_id")
    .orderBy(
        F.col("_bronze_ingestion_timestamp").desc()
    )
)

silver_df = (
    silver_df
    .withColumn(
        "_duplicate_rank",
        F.row_number().over(window_spec)
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "is_duplicate",
    F.col("_duplicate_rank") > 1
)

In [0]:
silver_df = silver_df.withColumn(
    "is_valid",
    F.col("is_valid") &
    (~F.col("is_duplicate"))
)

In [0]:
%skip
display(silver_df)

In [0]:
# ============================================================
# REJECTED RECORDS
# ============================================================

rejected_df = (
    silver_df
    .filter(~F.col("is_valid"))
)
print(
    f"Rejected records: {rejected_df.count()}"
)

In [0]:
# ============================================================
# CLEAN SILVER DATA
# ============================================================

clean_silver_df = (
    silver_df
    .filter(F.col("is_valid"))
)

In [0]:
%skip
display(clean_silver_df)

In [0]:
# ============================================================
# SILVER METADATA
# ============================================================

clean_silver_df = (
    clean_silver_df

    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_silver_processed_date",
        F.current_date()
    )
)
display(clean_silver_df)

In [0]:
columns_to_drop = (
    validation_columns +
    [
        "amount_valid",
        "is_valid",
        "rejection_reason",
        "is_duplicate",
        "_duplicate_rank"
    ]
)

clean_silver_df = clean_silver_df.drop(
    *columns_to_drop
)

In [0]:
# ============================================================
# WRITE SILVER
# ============================================================

(
    clean_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(
    f"Silver table written: {SILVER_TABLE}"
)

In [0]:
# ============================================================
# WRITE REJECTED RECORDS
# ============================================================

(
    rejected_df
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(REJECTED_TABLE)
)

print(
    f"Rejected records written: {REJECTED_TABLE}"
)

In [0]:
# ============================================================
# SILVER SUMMARY
# ============================================================

bronze_count = bronze_df.count()
silver_count = spark.table(SILVER_TABLE).count()
rejected_count = rejected_df.count()

print("=" * 60)
print("SILVER PROCESSING SUMMARY")
print("=" * 60)

print(f"Bronze records    : {bronze_count}")
print(f"Silver records    : {silver_count}")
print(f"Rejected records  : {rejected_count}")
print(f"Processed total   : {silver_count + rejected_count}")
print("=" * 60)

In [0]:
# ============================================================
# SOURCE QUALITY REPORT
# ============================================================

(
    silver_df
    .groupBy("data_source")
    .agg(
        F.count("*").alias("total_records"),

        F.sum(
            F.when(F.col("is_valid"), 1).otherwise(0)
        ).alias("valid_records"),

        F.sum(
            F.when(~F.col("is_valid"), 1).otherwise(0)
        ).alias("rejected_records")
    )
    .withColumn(
        "quality_percentage",
        F.round(
            F.col("valid_records") /
            F.col("total_records") * 100,
            2
        )
    )
    .orderBy("data_source")
    .show()
)